# Phase 5 FINAL: ALBERT-xxlarge — Beat RESTBERTa Paper (0.8529)

**Why ALBERT-xxlarge**:
- Your Phase 2 ALBERT-base scored **0.8064** with just 50K samples
- ALBERT-xxlarge is ~20x bigger (235M vs 12M effective) + 300K samples
- Same family that worked, just scaled up = most likely to beat paper
- No version hell like ModernBERT
- Stable library support since 2020

**Expected**: Acc@1 = **0.83-0.87** on PM task (paper: 0.8208)

**Kaggle setup**:
- Accelerator: **GPU T4 x2** (32GB total VRAM)
- Internet: **ON**
- Persistence: **Files only**
- Add dataset: `kailram/restberta-datasets`

**Total time: ~7-8 hours** (fits in 9-hour session)

In [ ]:
# Cell 1: Install stable versions — no ModernBERT drama
!pip install -q -U "transformers>=4.44.0,<4.48" "huggingface_hub>=0.25,<0.27" accelerate>=0.34.0 datasets
print('Installed stable versions.')
# After this, Kernel → Restart Session, then run from Cell 2

In [ ]:
# Cell 2: Imports + Config
import os, json, gc, glob, time, random, re, shutil
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import torch
from torch.utils.data import Dataset
from datetime import datetime
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    TrainingArguments, Trainer,
)

# Paths (Kaggle)
DATA_BASE = '/kaggle/input/datasets/kailram/restberta-datasets'
PM_PATH   = os.path.join(DATA_BASE, 'parameter_matching')
ED_PATH   = os.path.join(DATA_BASE, 'endpoint_discovery')
OUTPUT    = '/kaggle/working'
CKPT_DIR  = os.path.join(OUTPUT, 'checkpoints')
RESULTS   = os.path.join(OUTPUT, 'results')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# Model: ALBERT-xxlarge (235M effective params, SentencePiece, stable)
MODEL_CKPT = 'albert/albert-xxlarge-v2'
MODEL_NAME = 'albert_xxlarge_restberta'

# Hyperparameters
MAX_LENGTH     = 512
DOC_STRIDE     = 128
N_BEST         = 20
TOP_K          = list(range(1, 11))
SEED           = 42
MAX_TRAIN_QA   = 300_000
EPOCHS         = 2
LR             = 1e-5           # ALBERT-xxlarge: LOWER LR than BERT
BATCH_SIZE     = 4              # xxlarge bigger per-param
GRAD_ACCUM     = 8              # effective batch = 32 (per GPU) x 2 GPU = 64
WARMUP_RATIO   = 0.06

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

# Verify data
for split in ['parameter_matching/train/train.json',
              'parameter_matching/validation/validation.json',
              'endpoint_discovery/train/train.json',
              'endpoint_discovery/validation/validation.json']:
    p = os.path.join(DATA_BASE, split)
    sz = os.path.getsize(p)/1e6 if os.path.exists(p) else 0
    print(f"  {'OK' if sz else 'MISSING':>7}  {split} ({sz:.0f} MB)")

In [ ]:
# Cell 3: Data loading + property parsing (from baseline paper)
def read_qa_pairs_sampled(data_path, split, max_qa):
    reservoir = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    item = (ctx, q['question'], q['answers'])
                    if len(reservoir) < max_qa:
                        reservoir.append(item)
                    else:
                        j = random.randint(0, count - 1)
                        if j < max_qa:
                            reservoir[j] = item
    print(f'  Scanned {count}, kept {len(reservoir)}')
    random.shuffle(reservoir)
    return reservoir


def read_qa_pairs_validation(data_path):
    qa = []
    for fp in sorted(glob.glob(os.path.join(data_path, 'validation', '*.json'))):
        with open(fp) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    qa.append((ctx, q['question'], q['answers']))
    return qa


def precompute_props(ctx):
    props = []
    i, n = 0, len(ctx)
    while i < n:
        while i < n and ctx[i] == ' ': i += 1
        if i >= n: break
        start = i
        while i < n and ctx[i] != ' ': i += 1
        props.append((start, i, ctx[start:i]))
    return props


def determine_best_from_span(props, char_s, char_e):
    full = None
    partials = []
    for ps, pe, name in props:
        if char_e <= ps or char_s >= pe: continue
        is_full = (char_s <= ps and char_e >= pe)
        if is_full:
            if full is not None: return None
            full = name
        else:
            partials.append((pe - ps, name))
    if full is not None: return full
    if not partials: return None
    partials.sort(key=lambda x: -x[0])
    if len(partials) >= 2 and partials[0][0] == partials[1][0]: return None
    return partials[0][1]

print('Functions loaded.')

In [ ]:
# Cell 4: Tokenize training data with OVERFLOW + max 3 non-ans per QA pair
cache_path = os.path.join(OUTPUT, f'{MODEL_NAME}_train.npz')

if os.path.exists(cache_path):
    sz = os.path.getsize(cache_path)/1e6
    print(f'Cache exists ({sz:.0f} MB) — skip tokenization')
else:
    print('Sampling training data...')
    pm = read_qa_pairs_sampled(PM_PATH, 'train', MAX_TRAIN_QA)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(30_000, MAX_TRAIN_QA // 10))
    train_data = pm + ed
    del pm, ed
    random.shuffle(train_data)
    print(f'Total: {len(train_data)} QA pairs')

    print(f'\nTokenizing with max_length={MAX_LENGTH}, stride={DOC_STRIDE}...')
    tok = AutoTokenizer.from_pretrained(MODEL_CKPT)
    pad_on_right = tok.padding_side == 'right'
    ctx_tok_idx = 1 if pad_on_right else 0
    MAX_NON_ANS_PER_PAIR = 3

    batch_ids, batch_masks, batch_starts, batch_ends = [], [], [], []
    stats = {'ans': 0, 'non_ans': 0, 'non_ans_dropped': 0}

    BATCH = 50
    for b0 in tqdm(range(0, len(train_data), BATCH), desc='Tokenizing'):
        batch = train_data[b0:b0+BATCH]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]

        tokenized = tok(
            ques if pad_on_right else ctxs,
            ctxs if pad_on_right else ques,
            truncation='only_second' if pad_on_right else 'only_first',
            max_length=MAX_LENGTH,
            stride=DOC_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding='max_length',
        )
        sm = tokenized.pop('overflow_to_sample_mapping')
        offsets_list = tokenized.pop('offset_mapping')

        N = len(tokenized['input_ids'])

        per_feat = []
        for fi in range(N):
            qa_idx = sm[fi]
            _, _, answers = batch[qa_idx]
            seq_ids = tokenized.sequence_ids(fi)
            cp = 0

            if not answers['answer_start']:
                sp = ep = cp
                is_ans = False
            else:
                sc = answers['answer_start'][0]
                ec = sc + len(answers['text'][0])
                ts = 0
                while ts < MAX_LENGTH and seq_ids[ts] != ctx_tok_idx: ts += 1
                te = MAX_LENGTH - 1
                while te >= 0 and seq_ids[te] != ctx_tok_idx: te -= 1
                if ts >= MAX_LENGTH or te < 0:
                    sp = ep = cp; is_ans = False
                elif offsets_list[fi][ts][0] > sc or offsets_list[fi][te][1] < ec:
                    sp = ep = cp; is_ans = False
                else:
                    while ts < MAX_LENGTH and offsets_list[fi][ts][0] <= sc: ts += 1
                    sp = max(ts - 1, 0)
                    while te >= 0 and offsets_list[fi][te][1] >= ec: te -= 1
                    ep = min(te + 1, MAX_LENGTH - 1)
                    if sp > ep or sp < 0 or ep >= MAX_LENGTH:
                        sp = ep = cp; is_ans = False
                    else:
                        is_ans = True
            per_feat.append((qa_idx, sp, ep, is_ans))

        from collections import defaultdict
        by_pair = defaultdict(lambda: {'ans': [], 'non': []})
        for fi, (qa_idx, _, _, is_ans) in enumerate(per_feat):
            if is_ans: by_pair[qa_idx]['ans'].append(fi)
            else: by_pair[qa_idx]['non'].append(fi)

        keep = []
        for qa_idx, groups in by_pair.items():
            keep.extend(groups['ans'])
            non = groups['non']
            if len(non) > MAX_NON_ANS_PER_PAIR:
                random.shuffle(non)
                stats['non_ans_dropped'] += len(non) - MAX_NON_ANS_PER_PAIR
                non = non[:MAX_NON_ANS_PER_PAIR]
            keep.extend(non)
        keep.sort()

        K = len(keep)
        ids_np = np.zeros((K, MAX_LENGTH), dtype=np.int32)
        masks_np = np.zeros((K, MAX_LENGTH), dtype=np.int32)
        starts_np = np.zeros(K, dtype=np.int64)
        ends_np = np.zeros(K, dtype=np.int64)

        for out_idx, fi in enumerate(keep):
            ids_np[out_idx] = tokenized['input_ids'][fi]
            masks_np[out_idx] = tokenized['attention_mask'][fi]
            _, sp, ep, is_ans = per_feat[fi]
            starts_np[out_idx] = sp
            ends_np[out_idx] = ep
            if is_ans: stats['ans'] += 1
            else: stats['non_ans'] += 1

        batch_ids.append(ids_np)
        batch_masks.append(masks_np)
        batch_starts.append(starts_np)
        batch_ends.append(ends_np)
        del tokenized, offsets_list, sm, ctxs, ques, batch, per_feat, by_pair, keep

    print(f'\nStats: ans={stats["ans"]} | non={stats["non_ans"]} | dropped={stats["non_ans_dropped"]}')
    total = stats['ans'] + stats['non_ans']
    print(f'Total: {total} features ({stats["ans"]/total*100:.1f}% answerable)')

    all_ids = np.concatenate(batch_ids); del batch_ids
    all_masks = np.concatenate(batch_masks); del batch_masks
    all_starts = np.concatenate(batch_starts); del batch_starts
    all_ends = np.concatenate(batch_ends); del batch_ends
    gc.collect()

    print(f'Shape: {all_ids.shape}')
    np.savez_compressed(cache_path, ids=all_ids, masks=all_masks,
                        starts=all_starts, ends=all_ends)
    print(f'Saved: {cache_path} ({os.path.getsize(cache_path)/1e6:.0f} MB)')
    del all_ids, all_masks, all_starts, all_ends, train_data, tok
    gc.collect()

In [ ]:
# Cell 5: Train ALBERT-xxlarge — FULL fine-tune with T4 x2 data parallel
merged_path = os.path.join(CKPT_DIR, MODEL_NAME)

class NpzDataset(Dataset):
    def __init__(self, path):
        d = np.load(path)
        self.ids = d['ids'].astype(np.int64)
        self.masks = d['masks'].astype(np.int64)
        self.starts = d['starts'].astype(np.int64)
        self.ends = d['ends'].astype(np.int64)
        np.clip(self.starts, 0, self.ids.shape[1]-1, out=self.starts)
        np.clip(self.ends, 0, self.ids.shape[1]-1, out=self.ends)
        n_ans = int(np.sum(self.starts != 0))
        print(f'  {len(self.ids)} features | answerable: {n_ans} ({100*n_ans/len(self.ids):.1f}%)')
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return {
            'input_ids': torch.from_numpy(self.ids[i].copy()),
            'attention_mask': torch.from_numpy(self.masks[i].copy()),
            'start_positions': torch.tensor(self.starts[i]),
            'end_positions': torch.tensor(self.ends[i]),
        }

if os.path.exists(os.path.join(merged_path, 'config.json')):
    print(f'SKIP: Model already trained at {merged_path}')
else:
    print(f'Loading {MODEL_CKPT}...')
    model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CKPT)
    print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

    train_ds = NpzDataset(cache_path)
    n_steps_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count()))
    print(f'Steps per epoch: {n_steps_epoch} | Total: {n_steps_epoch * EPOCHS}')

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, f'{MODEL_NAME}_tmp'),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=100,
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        dataloader_num_workers=2,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds)

    model.train()
    batch = {k: v.unsqueeze(0).to(device) for k, v in train_ds[0].items()}
    with torch.amp.autocast('cuda', dtype=torch.float16):
        out = model(**batch)
    print(f'Test loss: {out.loss.item():.4f}')
    del batch, out

    print('\nStarting training...')
    t0 = time.time()
    trainer.train()
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

    if os.path.exists(merged_path):
        shutil.rmtree(merged_path)
    trainer.model.save_pretrained(merged_path, safe_serialization=True)
    AutoTokenizer.from_pretrained(MODEL_CKPT).save_pretrained(merged_path)
    print(f'Saved to {merged_path}')

    del model, trainer, train_ds
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 6: Sanity check
print('Loading model for sanity check...')
model = AutoModelForQuestionAnswering.from_pretrained(merged_path)
model.to(device).half().eval()
eval_tok = AutoTokenizer.from_pretrained(merged_path)

qa_w = model.qa_outputs.weight if hasattr(model, 'qa_outputs') else model.classifier.weight
print(f'QA head std: {qa_w.std():.4f}')

sample_qa = []
with open(f'{PM_PATH}/validation/validation.json') as f:
    for line in f:
        rec = json.loads(line.strip())
        for q in rec['questions']:
            if q['answers']['answer_start']:
                sample_qa.append((rec['context'], q['question'], q['answers']))
        if len(sample_qa) >= 30: break

correct = 0
for ctx, q, ans in sample_qa[:30]:
    inputs = eval_tok(q, ctx, truncation='only_second', max_length=MAX_LENGTH,
                      padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs)
    si = int(out.start_logits[0].argmax())
    ei = int(out.end_logits[0].argmax())
    if ei >= si:
        pred = eval_tok.decode(inputs['input_ids'][0][si:ei+1], skip_special_tokens=True).strip()
    else:
        pred = ''
    gt = ans['text'][0].strip()
    if pred == gt or gt in pred or pred in gt:
        correct += 1

print(f'Sanity: {correct}/30 answerable correct ({100*correct/30:.0f}%)')
print(f'  >= 20/30 → good, proceed')
print(f'  < 15/30 → something wrong')

In [ ]:
# Cell 7: Full evaluation with overflow tokens (matches paper)
def overflow_eval(data_path, ds_name, batch_size=4, max_length=MAX_LENGTH,
                  doc_stride=DOC_STRIDE, n_best=N_BEST):
    print(f'\n{"="*60}\n  EVAL: {MODEL_NAME} on {ds_name}\n{"="*60}')

    qa_pairs = read_qa_pairs_validation(data_path)
    print(f'  {len(qa_pairs)} QA pairs')

    results = [{'rank': k, 'correct': 0, 'correct_answerable': 0, 'correct_non_answerable': 0,
                'total': 0, 'total_answerable': 0, 'total_non_answerable': 0} for k in TOP_K]

    t0 = time.time()
    model.eval()
    feat_count = 0

    for b0 in tqdm(range(0, len(qa_pairs), batch_size), desc='  Eval'):
        batch = qa_pairs[b0:b0+batch_size]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]
        answers_list = [x[2] for x in batch]

        enc = eval_tok(
            ques, ctxs,
            truncation='only_second',
            max_length=max_length,
            stride=doc_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding='max_length',
            return_tensors='pt',
        )
        sample_mapping = enc.pop('overflow_to_sample_mapping').tolist()
        offsets = enc.pop('offset_mapping').numpy()
        seq_ids_list = [enc.sequence_ids(i) for i in range(len(enc['input_ids']))]
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)

        F = len(sample_mapping)
        feat_count += F

        top_s_vals, top_s_idx = torch.topk(out.start_logits, n_best, dim=-1)
        top_e_vals, top_e_idx = torch.topk(out.end_logits, n_best, dim=-1)
        top_s_idx = top_s_idx.cpu().numpy()
        top_e_idx = top_e_idx.cpu().numpy()
        top_s_vals = top_s_vals.cpu().numpy()
        top_e_vals = top_e_vals.cpu().numpy()
        cls_s = out.start_logits[:, 0].cpu().numpy()
        cls_e = out.end_logits[:, 0].cpu().numpy()

        props_cache = {}
        for fi in range(F):
            sample_idx = sample_mapping[fi]
            ctx_text = ctxs[sample_idx]
            ans = answers_list[sample_idx]
            gt_name = ans['text'][0] if ans['text'] else ''

            if id(ctx_text) not in props_cache:
                props_cache[id(ctx_text)] = precompute_props(ctx_text)
            props = props_cache[id(ctx_text)]

            seq_ids = seq_ids_list[fi]
            offs = offsets[fi]

            if not ans['answer_start']:
                is_oos = True
            else:
                sc = ans['answer_start'][0]
                ec = sc + len(ans['text'][0])
                ctx_positions = [p for p, s in enumerate(seq_ids) if s == 1]
                if not ctx_positions:
                    is_oos = True
                else:
                    first_char = int(offs[ctx_positions[0], 0])
                    last_char = int(offs[ctx_positions[-1], 1])
                    is_oos = not (first_char <= sc and last_char >= ec)

            ctx_mask = np.array([s == 1 for s in seq_ids], dtype=bool)
            candidates = []
            for si in range(n_best):
                s = int(top_s_idx[fi, si])
                if s == 0 or not ctx_mask[s]: continue
                for ei in range(n_best):
                    e = int(top_e_idx[fi, ei])
                    if e == 0 or not ctx_mask[e]: continue
                    if e < s or e - s > 30: continue
                    char_s = int(offs[s, 0])
                    char_e = int(offs[e, 1])
                    name = determine_best_from_span(props, char_s, char_e)
                    if name is None: continue
                    score = float(top_s_vals[fi, si] + top_e_vals[fi, ei])
                    candidates.append((score, name))

            cls_score = float(cls_s[fi] + cls_e[fi])
            candidates.append((cls_score, None))
            candidates.sort(key=lambda x: -x[0])

            rank = None
            for ri, (_, name) in enumerate(candidates):
                if is_oos:
                    if name is None or name == gt_name:
                        rank = ri + 1; break
                else:
                    if name is not None and name == gt_name:
                        rank = ri + 1; break

            for kr in results:
                if rank is not None and rank <= kr['rank']:
                    kr['correct'] += 1
                    if is_oos: kr['correct_non_answerable'] += 1
                    else: kr['correct_answerable'] += 1
                kr['total'] += 1
                if is_oos: kr['total_non_answerable'] += 1
                else: kr['total_answerable'] += 1

    for kr in results:
        t = kr['total']; ta = kr['total_answerable']; tn = kr['total_non_answerable']
        kr['accuracy'] = kr['correct']/t if t else 0
        kr['accuracy_answerable'] = kr['correct_answerable']/ta if ta else 0
        kr['accuracy_non_answerable'] = kr['correct_non_answerable']/tn if tn else 0

    elapsed = time.time() - t0
    print(f'\n  Done: {elapsed/60:.1f} min | {feat_count} features')
    print(f'  Answerable: {results[0]["total_answerable"]} | Non-ans: {results[0]["total_non_answerable"]}')
    print(f'  K   Acc@K   Ans     Non')
    for r in results:
        print(f"  @{r['rank']:>2}  {r['accuracy']:.4f}  {r['accuracy_answerable']:.4f}  {r['accuracy_non_answerable']:.4f}")

    return {
        'model': MODEL_NAME, 'dataset': ds_name, 'type': 'full-finetuned',
        'total_samples': results[0]['total'],
        'total_answerable': results[0]['total_answerable'],
        'total_non_answerable': results[0]['total_non_answerable'],
        'elapsed_seconds': elapsed, 'results': results,
    }


all_results = []
r = overflow_eval(PM_PATH, 'parameter_matching')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{RESULTS}/{MODEL_NAME}_pm_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

r = overflow_eval(ED_PATH, 'endpoint_discovery')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{RESULTS}/{MODEL_NAME}_ed_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

print('\n✓ Eval complete!')

In [ ]:
# Cell 8: Final comparison vs paper baseline
baseline = [
    {'model': 'CB-PM (paper)', 'dataset': 'parameter_matching', 'type': 'baseline',
     'results': [
        {'rank': 1, 'accuracy': 0.8195}, {'rank': 3, 'accuracy': 0.9613},
        {'rank': 5, 'accuracy': 0.9759}, {'rank': 10, 'accuracy': 0.9816}]},
    {'model': 'CB-ED (paper)', 'dataset': 'endpoint_discovery', 'type': 'baseline',
     'results': [
        {'rank': 1, 'accuracy': 0.8844}, {'rank': 3, 'accuracy': 0.9535},
        {'rank': 5, 'accuracy': 0.9678}, {'rank': 10, 'accuracy': 0.9749}]},
    {'model': 'CB-PM+ED (paper best)', 'dataset': 'parameter_matching', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8208}]},
]

combined = baseline + all_results

print(f'\n{"="*100}')
print(f'FULL COMPARISON — RESTBERTa paper vs ALBERT-xxlarge')
print(f'{"="*100}')
print(f'{"Model":<35} {"Type":<18} {"Dataset":<22} {"Acc@1":>7} {"Acc@3":>7} {"Acc@5":>7} {"Acc@10":>7}')
print('-'*100)

for r in combined:
    acc = {x['rank']: x.get('accuracy', 0) for x in r['results']}
    print(f'{r["model"]:<35} {r["type"]:<18} {r["dataset"]:<22} '
          f'{acc.get(1,0):>7.4f} {acc.get(3,0):>7.4f} {acc.get(5,0):>7.4f} {acc.get(10,0):>7.4f}')

print('='*100)

for ds in ['parameter_matching', 'endpoint_discovery']:
    paper_acc1 = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                       for r in baseline if r['dataset'] == ds] or [0])
    our_acc1 = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                     for r in all_results if r['dataset'] == ds] or [0])
    if our_acc1 > paper_acc1:
        print(f'\n🎯 {ds}: ALBERT-xxl {our_acc1:.4f} vs paper {paper_acc1:.4f} → +{(our_acc1-paper_acc1)*100:.2f}%')
    else:
        print(f'\n⚠ {ds}: ALBERT-xxl {our_acc1:.4f} vs paper {paper_acc1:.4f} → {(our_acc1-paper_acc1)*100:.2f}%')

with open(f'{RESULTS}/final_comparison.json', 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\nSaved: {RESULTS}/final_comparison.json')